# Clase 8 — Protocolos Criptográficos: Diffie-Hellman, Estaciones de Confianza, PKI/X.509 y TLS 1.3

> **Curso:** Criptografía Aplicada  
> **Objetivo general:** Entender cómo se construye seguridad real en Internet combinando protocolos de intercambio de claves, infraestructura de confianza y autenticación mediante certificados.

---

## Tabla de contenidos

1. [Introducción: de algoritmos a protocolos](#1)
2. [Diffie-Hellman clásico (DH)](#2)
3. [Implementación práctica de DH en Python](#3)
4. [Ataque MITM y por qué DH solo no autentica](#4)
5. [Estaciones de confianza](#5)
6. [PKI (Infraestructura de Clave Pública)](#6)
7. [Certificados X.509 en detalle](#7)
8. [Práctica: inspección de certificados en Python](#8)
9. [TLS 1.3: handshake en detalle](#9)
10. [Práctica: derivación de claves (HKDF) estilo TLS 1.3](#10)
11. [0-RTT, PFS, reanudación y seguridad operativa](#11)
12. [Buenas prácticas de despliegue](#12)
13. [Temas adicionales pertinentes](#13)
14. [Ejercicios propuestos](#14)
15. [Referencias](#15)


<a id='1'></a>
## 1) Introducción: de algoritmos a protocolos

Un algoritmo criptográfico (AES, SHA-256, ECDSA) **no resuelve por sí solo** un problema de seguridad completo.  
La seguridad real surge de protocolos que integran:

- **Confidencialidad** (cifrado),
- **Integridad** (MAC/AEAD),
- **Autenticación** (firmas/certificados),
- **Gestión de claves** (intercambio, rotación, revocación),
- **Modelo de confianza** (quién certifica a quién).

### Objetivo técnico de esta clase

Conectar los bloques conceptuales en un flujo coherente:

- DH/ECDHE permite acordar claves efímeras.
- PKI + X.509 permite autenticar la identidad del servidor.
- TLS 1.3 combina ambos para lograr canal seguro con **Perfect Forward Secrecy (PFS)**.


<a id='2'></a>
## 2) Diffie-Hellman clásico (DH)

### 2.1 Idea fundamental

Dos partes (Alice y Bob) acuerdan un secreto compartido sobre canal inseguro sin enviarlo directamente.

Parámetros públicos:

- primo grande \(p\)
- generador \(g\) del subgrupo

Secretos privados:

- Alice elige \(a\), Bob elige \(b\)

Mensajes:

- Alice envía \(A = g^a \mod p\)
- Bob envía \(B = g^b \mod p\)

Secreto compartido:

\[
K = B^a \equiv A^b \equiv g^{ab} \mod p
\]

### 2.2 Seguridad

Se apoya en el **problema del logaritmo discreto** en grupos finitos.

### 2.3 Riesgos si se implementa mal

- Grupos débiles o pequeños.
- Falta de validación de claves públicas.
- Uso de DH estático sin autenticación.
- Vulnerabilidad a **Man-in-the-Middle (MITM)**.


## Importaciones globales


In [ ]:
import secrets
import hashlib
import hmac
import math
from dataclasses import dataclass
from typing import Tuple

print("Entorno listo ✓")


<a id='3'></a>
## 3) Implementación práctica de DH en Python

> **Nota pedagógica:** Los primos pequeños aquí son para aprendizaje; en producción se usan grupos estandarizados (RFC 7919) o ECDHE/X25519.


In [ ]:
# Utilidades matemáticas para DH educativo

def modexp(base: int, exp: int, mod: int) -> int:
    return pow(base, exp, mod)

def inv_mod(a: int, p: int) -> int:
    return pow(a, -1, p)

def is_probable_prime(n: int, k: int = 8) -> bool:
    """Miller-Rabin básico para demostración."""
    if n < 2:
        return False
    small = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
    for p in small:
        if n % p == 0:
            return n == p

    d = n - 1
    s = 0
    while d % 2 == 0:
        s += 1
        d //= 2

    for _ in range(k):
        a = secrets.randbelow(n - 3) + 2
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        witness = True
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                witness = False
                break
        if witness:
            return False
    return True

print("Utilidades definidas ✓")


In [ ]:
# Ejemplo DH de juguete (pequeño y vulnerable, solo didáctico)
p = 2089           # primo pequeño para ver números manejables
g = 2

assert is_probable_prime(p), "p debe ser primo"

# Secretos privados
alice_priv = secrets.randbelow(p - 2) + 2
bob_priv = secrets.randbelow(p - 2) + 2

# Públicos
A = modexp(g, alice_priv, p)
B = modexp(g, bob_priv, p)

# Secreto compartido
K_alice = modexp(B, alice_priv, p)
K_bob = modexp(A, bob_priv, p)

print(f"p={p}, g={g}")
print(f"Alice privado: {alice_priv}")
print(f"Bob privado:   {bob_priv}")
print(f"A=g^a mod p:  {A}")
print(f"B=g^b mod p:  {B}")
print(f"K Alice:      {K_alice}")
print(f"K Bob:        {K_bob}")
print("¿Coinciden?", K_alice == K_bob)


In [ ]:
# Derivación de clave simétrica desde K (KDF simple didáctica)

def kdf_sha256(shared_secret: int, info: bytes = b"demo-dh") -> bytes:
    raw = shared_secret.to_bytes((shared_secret.bit_length() + 7) // 8 or 1, "big")
    return hashlib.sha256(raw + b"|" + info).digest()

k_session = kdf_sha256(K_alice, b"clase8")
print("Clave de sesión (SHA-256):", k_session.hex())
print("Longitud:", len(k_session), "bytes")


### 3.1 Validación básica de parámetros

En despliegues reales no basta con recibir \(A\) y \(B\); se valida que estén en rango y pertenezcan al grupo/subgrupo correcto para evitar ataques de subgrupo pequeño.


In [ ]:
def validate_dh_public(pub: int, p: int) -> bool:
    # Reglas mínimas educativas (no suficientes para producción)
    if not (2 <= pub <= p - 2):
        return False
    return True

print("A válido:", validate_dh_public(A, p))
print("B válido:", validate_dh_public(B, p))
print("Valor inválido 1:", validate_dh_public(1, p))
print("Valor inválido p-1:", validate_dh_public(p-1, p))


### 3.2 Mini-demostración de ataque en grupo pequeño

Si el grupo es pequeño, el logaritmo discreto se puede romper por fuerza bruta.


In [ ]:
def brute_force_dlog(g: int, target: int, p: int, limit: int = None) -> int | None:
    """Recupera x por fuerza bruta tal que g^x ≡ target (mod p), o None si no existe."""
    if limit is None:
        limit = p
    cur = 1
    for x in range(limit):
        if cur == target:
            return x
        cur = (cur * g) % p
    return None

recovered_a = brute_force_dlog(g, A, p)
recovered_b = brute_force_dlog(g, B, p)
print("a recuperado:", recovered_a)
print("b recuperado:", recovered_b)
print("Ataque exitoso en grupo pequeño:", recovered_a == alice_priv and recovered_b == bob_priv)


<a id='4'></a>
## 4) Ataque MITM y por qué DH solo no autentica

DH asegura acuerdo de clave, pero no autenticidad de interlocutor.  
Un atacante activo puede interponerse y establecer dos claves distintas:

- Alice ↔ Mallory
- Mallory ↔ Bob

Resultado: Mallory descifra, modifica y re-cifra tráfico.


In [ ]:
# Simulación simplificada de MITM sobre DH
p = 2089
g = 2

a = secrets.randbelow(p - 2) + 2
b = secrets.randbelow(p - 2) + 2
m1 = secrets.randbelow(p - 2) + 2
m2 = secrets.randbelow(p - 2) + 2

A = pow(g, a, p)      # Alice -> "Bob"
B = pow(g, b, p)      # Bob -> "Alice"
M_to_bob = pow(g, m1, p)
M_to_alice = pow(g, m2, p)

# Alice y Bob creen hablar entre sí, pero usan claves con Mallory
K_alice = pow(M_to_alice, a, p)
K_bob = pow(M_to_bob, b, p)
K_mallory_with_alice = pow(A, m2, p)
K_mallory_with_bob = pow(B, m1, p)

print("K_alice == K_mallory_with_alice:", K_alice == K_mallory_with_alice)
print("K_bob   == K_mallory_with_bob:  ", K_bob == K_mallory_with_bob)
print("Alice y Bob comparten misma clave?", K_alice == K_bob)


**Conclusión:** DH necesita autenticación adicional (firmas, certificados, PSK autenticado, etc.).  
Aquí es donde entran PKI, X.509 y TLS.


<a id='5'></a>
## 5) Estaciones de confianza

En ecosistemas reales aparecen múltiples "terceros confiables":

1. **CA (Certification Authority):** emite certificados X.509.
2. **RA (Registration Authority):** valida identidad antes de emisión.
3. **TSA (Time-Stamping Authority):** sellado temporal confiable.
4. **OCSP Responder:** estado de revocación en línea.
5. **CT Logs (Certificate Transparency):** auditoría pública de certificados emitidos.
6. **KDC (en otros protocolos, como Kerberos):** centro de distribución de claves simétricas.

### Modelo de confianza

- El cliente confía en un **trust store** de raíces.
- Valida cadena `leaf -> intermedia(s) -> raíz`.
- Verifica nombre (SAN), vigencia, uso de clave y revocación.


<a id='6'></a>
## 6) PKI (Infraestructura de Clave Pública)

La PKI es la combinación de:

- Políticas y procesos,
- Autoridades (CA/RA),
- Certificados y listas de revocación,
- Almacenamiento de claves y auditoría.

### 6.1 Ciclo de vida de un certificado

1. Generación de clave privada + CSR.
2. Validación de identidad (DV/OV/EV o interna corporativa).
3. Emisión por CA intermedia.
4. Instalación y uso en servidor.
5. Renovación o revocación.

### 6.2 Cadena de confianza

- **Root CA**: auto-firmada, offline idealmente.
- **Intermediate CA**: firma certificados operativos.
- **Leaf cert**: certificado de servidor/usuario.

### 6.3 Fallas comunes de PKI

- Falta de rotación de claves.
- Uso de algoritmos deprecados (SHA-1, RSA pequeño).
- No monitorear CT logs.
- No gestionar revocación eficazmente.


<a id='7'></a>
## 7) Certificados X.509 en detalle

Un certificado X.509 v3 contiene (resumen):

- **Version** (v3)
- **Serial Number**
- **Signature Algorithm**
- **Issuer**
- **Validity** (`notBefore`, `notAfter`)
- **Subject**
- **Subject Public Key Info**
- **Extensions** (clave en seguridad moderna)

### Extensiones críticas relevantes

- **basicConstraints**: define si es CA.
- **keyUsage**: firma, cifrado de clave, etc.
- **extendedKeyUsage**: serverAuth, clientAuth, codeSigning...
- **subjectAltName (SAN)**: DNS/IP válidos (reemplaza CN para hostname).
- **authorityKeyIdentifier / subjectKeyIdentifier**.
- **CRL Distribution Points / AIA (OCSP)**.

### Validación de hostname

Hoy el navegador valida contra SAN (DNS/IP). El CN histórico no es suficiente.


<a id='8'></a>
## 8) Práctica: inspección de certificados en Python

Esta práctica usa la librería `cryptography` si está disponible.  
Si no está instalada, el notebook no falla: imprime instrucciones.


In [ ]:
try:
    from cryptography import x509
    from cryptography.hazmat.primitives import serialization
    from cryptography.hazmat.backends import default_backend
    import ssl, socket
    CRYPTO_OK = True
except Exception as e:
    CRYPTO_OK = False
    print("cryptography no disponible:", e)

print("cryptography disponible:", CRYPTO_OK)


In [ ]:
# Descarga de certificado de un sitio (opcional, requiere red)

def fetch_server_cert_pem(host: str, port: int = 443, timeout: int = 5) -> bytes:
    """Descarga el certificado TLS del servidor en formato PEM.

    Puede lanzar `socket.timeout`, `ssl.SSLError` u `OSError` si hay fallos de red/TLS.
    """
    ctx = ssl.create_default_context()
    with socket.create_connection((host, port), timeout=timeout) as sock:
        with ctx.wrap_socket(sock, server_hostname=host) as ssock:
            der = ssock.getpeercert(binary_form=True)
    pem = ssl.DER_cert_to_PEM_cert(der).encode()
    return pem

if CRYPTO_OK:
    try:
        pem = fetch_server_cert_pem("example.com")
        cert = x509.load_pem_x509_certificate(pem, default_backend())
        print("Subject:", cert.subject)
        print("Issuer:", cert.issuer)
        print("Serial:", cert.serial_number)
        print("Validez:", cert.not_valid_before, "->", cert.not_valid_after)
    except Exception as e:
        print("No se pudo inspeccionar certificado en línea (entorno restringido):", e)


In [ ]:
# Extraer SAN y extensiones clave si se obtuvo certificado
if CRYPTO_OK:
    try:
        san = cert.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
        print("SAN DNS:", san.get_values_for_type(x509.DNSName)[:10])
    except Exception as e:
        print("SAN no disponible:", e)

    wanted = [
        x509.BasicConstraints,
        x509.KeyUsage,
        x509.ExtendedKeyUsage,
        x509.AuthorityKeyIdentifier,
        x509.SubjectKeyIdentifier,
    ]
    for ext_cls in wanted:
        try:
            ext = cert.extensions.get_extension_for_class(ext_cls)
            print(f"{ext_cls.__name__}:", ext.value)
        except Exception:
            pass


<a id='9'></a>
## 9) TLS 1.3 — Handshake en detalle

TLS 1.3 (RFC 8446) simplifica y fortalece TLS 1.2:

- Elimina suites débiles y RSA key exchange.
- Usa (EC)DHE para PFS.
- Reduce latencia (1-RTT en handshake completo).
- Cifra más temprano en el flujo.

### 9.1 Flujo típico (1-RTT)

1. **ClientHello**
   - versiones soportadas
   - cipher suites (AEAD + hash)
   - `key_share` (X25519/P-256)
   - `supported_groups`, `signature_algorithms`, SNI, ALPN
2. **ServerHello**
   - suite elegida
   - `key_share` del servidor
   - desde aquí ambos derivan `handshake secrets`
3. **EncryptedExtensions**
4. **Certificate**
5. **CertificateVerify** (firma transcript hash con clave privada del cert)
6. **Finished** (prueba de posesión de claves handshake)
7. Cliente valida certificado + firma + Finished del servidor
8. Cliente envía su **Finished**
9. Ambos derivan **application traffic keys**

### 9.2 Derivación de secretos (vista simplificada)

- `early_secret`
- `handshake_secret`
- `master_secret`
- `client/server handshake traffic secret`
- `client/server application traffic secret`

Todo se basa en HKDF (Extract + Expand) y transcript hash acumulado.


### 9.3 ¿Dónde autentica TLS al servidor?

En `Certificate + CertificateVerify`:

- El cliente verifica la cadena X.509 y hostname.
- Verifica la firma de `CertificateVerify` sobre el transcript.
- Así asegura que quien respondió en ECDHE posee la clave privada del certificado legítimo.

Esto mitiga MITM (salvo compromisos de CA, claves o dispositivo cliente).


<a id='10'></a>
## 10) Práctica: derivación de claves estilo TLS 1.3 (simplificada)

Implementaremos `HKDF-Extract`, `HKDF-Expand` y un `HKDF-Expand-Label` simplificado para visualizar cómo se derivan secretos encadenados.


In [ ]:
def hkdf_extract(salt: bytes, ikm: bytes, hashmod=hashlib.sha256) -> bytes:
    """HKDF-Extract (RFC 5869): mezcla IKM con sal para producir PRK."""
    return hmac.new(salt, ikm, hashmod).digest()

def hkdf_expand(prk: bytes, info: bytes, length: int, hashmod=hashlib.sha256) -> bytes:
    """HKDF-Expand (RFC 5869): expande PRK a la longitud solicitada."""
    out = b""
    t = b""
    counter = 1
    while len(out) < length:
        t = hmac.new(prk, t + info + bytes([counter]), hashmod).digest()
        out += t
        counter += 1
    return out[:length]

def hkdf_expand_label(secret: bytes, label: str, context: bytes, length: int, hashmod=hashlib.sha256) -> bytes:
    """Versión simplificada de HKDF-Expand-Label usada para ilustrar TLS 1.3."""
    full_label = b"tls13 " + label.encode()
    # Estructura inspirada en RFC 8446 (simplificada)
    hkdf_label = (
        length.to_bytes(2, "big") +
        bytes([len(full_label)]) + full_label +
        bytes([len(context)]) + context
    )
    return hkdf_expand(secret, hkdf_label, length, hashmod)

print("Funciones HKDF listas ✓")


In [ ]:
# Simulación de key schedule (didáctica, no interoperable exacta RFC)

hashmod = hashlib.sha256
zeros = b"\x00" * hashmod().digest_size
psk = b""  # no PSK en este ejemplo
shared_ecdh = secrets.token_bytes(32)  # en real viene de (EC)DHE

client_hello = b"ClientHello(...)"
server_hello = b"ServerHello(...)"
transcript_ch_sh = hashmod(client_hello + server_hello).digest()

early_secret = hkdf_extract(zeros, psk, hashmod)
derived_early = hkdf_expand_label(early_secret, "derived", b"", hashmod().digest_size, hashmod)
handshake_secret = hkdf_extract(derived_early, shared_ecdh, hashmod)

c_hs_traffic = hkdf_expand_label(handshake_secret, "c hs traffic", transcript_ch_sh, 32, hashmod)
s_hs_traffic = hkdf_expand_label(handshake_secret, "s hs traffic", transcript_ch_sh, 32, hashmod)

derived_hs = hkdf_expand_label(handshake_secret, "derived", b"", hashmod().digest_size, hashmod)
master_secret = hkdf_extract(derived_hs, b"", hashmod)

transcript_full = hashmod(client_hello + server_hello + b"EncryptedExtensions" + b"Certificate" + b"CertificateVerify" + b"Finished").digest()
c_ap_traffic = hkdf_expand_label(master_secret, "c ap traffic", transcript_full, 32, hashmod)
s_ap_traffic = hkdf_expand_label(master_secret, "s ap traffic", transcript_full, 32, hashmod)

print("early_secret       ", early_secret.hex())
print("handshake_secret   ", handshake_secret.hex())
print("master_secret      ", master_secret.hex())
print("c_hs_traffic       ", c_hs_traffic.hex())
print("s_hs_traffic       ", s_hs_traffic.hex())
print("c_ap_traffic       ", c_ap_traffic.hex())
print("s_ap_traffic       ", s_ap_traffic.hex())


### 10.1 Observación clave

Si cambia el transcript (por ejemplo por manipulación MITM), cambian los secrets derivados y fallan los mensajes `Finished`.  
Por eso TLS 1.3 "ata" criptográficamente todo el handshake.


<a id='11'></a>
## 11) 0-RTT, PFS, reanudación y seguridad operativa

### 11.1 Perfect Forward Secrecy (PFS)

Con ECDHE efímero, comprometer la clave privada del certificado en el futuro **no** revela sesiones pasadas.

### 11.2 Reanudación de sesión

TLS 1.3 usa PSK/tickets para reanudar y reducir latencia.

### 11.3 0-RTT

Permite enviar datos en el primer vuelo, pero:

- no tiene PFS total para esos datos iniciales,
- es vulnerable a replay a nivel aplicación.

Por eso debe habilitarse solo para operaciones idempotentes.


<a id='12'></a>
## 12) Buenas prácticas de despliegue

1. Priorizar **TLS 1.3** y deshabilitar protocolos obsoletos.
2. Preferir grupos **X25519** y/o P-256.
3. Certificados con claves fuertes (ECDSA P-256 o RSA 3072+ si aplica).
4. Renovación automática (ACME) y monitoreo de expiración.
5. Activar OCSP stapling y revisar CT logs.
6. Configurar HSTS en HTTPS público.
7. Rotar claves y auditar dependencias criptográficas.
8. Evitar criptografía casera en producción.


<a id='13'></a>
## 13) Temas adicionales pertinentes

### 13.1 mTLS (autenticación mutua)

Además de autenticar servidor, el servidor autentica cliente por certificado. Común en entornos B2B y service mesh.

### 13.2 QUIC + TLS 1.3

HTTP/3 integra TLS 1.3 sobre QUIC (UDP), reduciendo latencia y mejorando recuperación frente a pérdida.

### 13.3 Criptografía post-cuántica

Se están desplegando handshakes híbridos (ECDHE + KEM PQC) para transición gradual.


<a id='14'></a>
## 14) Ejercicios propuestos

1. Modifica el ejemplo DH para usar un primo mayor y mide tiempos.
2. Implementa verificación estricta de elementos de subgrupo para DH clásico.
3. Simula un flujo de validación de cadena X.509 con certificados locales.
4. Reproduce el key schedule simplificado cambiando el transcript y compara secretos.
5. Investiga una mala emisión histórica de certificados y explica impacto.
6. Diseña política mínima TLS para una API pública y justifica cada decisión.


<a id='15'></a>
## 15) Referencias

### Estándares y RFC

- RFC 8446 — *The Transport Layer Security (TLS) Protocol Version 1.3*  
  https://datatracker.ietf.org/doc/html/rfc8446
- RFC 5869 — *HMAC-based Extract-and-Expand Key Derivation Function (HKDF)*  
  https://datatracker.ietf.org/doc/html/rfc5869
- RFC 7919 — *Negotiated Finite Field Diffie-Hellman Ephemeral Parameters for TLS*  
  https://datatracker.ietf.org/doc/html/rfc7919
- RFC 5280 — *Internet X.509 Public Key Infrastructure Certificate and CRL Profile*  
  https://datatracker.ietf.org/doc/html/rfc5280
- RFC 6962 — *Certificate Transparency*  
  https://datatracker.ietf.org/doc/html/rfc6962

### Guías técnicas

- NIST SP 800-56A Rev.3 — *Recommendation for Pair-Wise Key-Establishment Schemes*  
- NIST SP 800-57 Part 1 Rev.5 — *Key Management Guidelines*  
- OWASP Transport Layer Protection Cheat Sheet  
  https://cheatsheetseries.owasp.org/
- Mozilla SSL Configuration Guidelines  
  https://ssl-config.mozilla.org/

### Bibliografía

- Menezes, van Oorschot, Vanstone — *Handbook of Applied Cryptography*
- Katz & Lindell — *Introduction to Modern Cryptography*
- Rescorla — *SSL and TLS: Designing and Building Secure Systems*
